In [3]:
import os, subprocess
from kaggle_secrets import UserSecretsClient

USER  = "shashiranjan-jnu"      # replace
EMAIL = "shashi17_scs@jnu.ac.in"    # replace, the email linked to GitHub
REPO  = "dpfl-braintumor"

token = UserSecretsClient().get_secret("kaggle_dpfl-braintumor")
url = f"https://{token}@github.com/{USER}/{REPO}.git"

def run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(r.stderr.replace(token, "***"))
    return r.stdout.replace(token, "***")

# 1. Fresh clone of main
os.chdir("/kaggle/working")
run(["rm", "-rf", REPO])
run(["git", "clone", "-q", url, REPO])
os.chdir(REPO)

# 2. Folders that only need a placeholder
keep_dirs = ["configs/experiments", "configs/grids", "data_manifest",
             "notebooks/kaggle", "results/runs", "results/tables",
             "figures", "paper", "scripts", "tests"]
for d in keep_dirs:
    os.makedirs(d, exist_ok=True)
    open(f"{d}/.gitkeep", "a").close()

# 3. Python packages
pkgs = ["src", "src/data", "src/models", "src/fl", "src/privacy",
        "src/xai", "src/attacks", "src/eval"]
for p in pkgs:
    os.makedirs(p, exist_ok=True)
    open(f"{p}/__init__.py", "a").close()

# 4. Documentation starters, never overwriting existing files
os.makedirs("docs", exist_ok=True)
starters = {
    "CHANGELOG.md": "# Changelog\n\nNewest entry first.\n",
    "docs/decisions.md": "# Design Decisions\n",
    "docs/lab_notebook.md": "# Lab Notebook\n",
    "docs/kaggle_setup.md": "# Kaggle Setup\n\nQuota check date and limits go here.\n",
}
for path, text in starters.items():
    if not os.path.exists(path):
        with open(path, "w") as f:
            f.write(text)

# 5. Extend .gitignore once
extra = """
# Project specific
data/
*.npy
*.npz
*.pt
*.pth
*.ckpt
checkpoints/
wandb/
.env
*.zip
.ipynb_checkpoints/
"""
gi = open(".gitignore").read() if os.path.exists(".gitignore") else ""
if "# Project specific" not in gi:
    with open(".gitignore", "a") as f:
        f.write(extra)

# 6. Commit and push
run(["git", "config", "user.name", USER])
run(["git", "config", "user.email", EMAIL])
run(["git", "add", "-A"])
if run(["git", "status", "--porcelain"]).strip():
    run(["git", "commit", "-q", "-m", "Scaffold repository structure"])
    run(["git", "push", "-q", "origin", "main"])
    print("Scaffold pushed to main.")
else:
    print("Nothing new to commit, scaffold already present.")

Scaffold pushed to main.
